In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
from pathlib import Path

p = Path("/content/drive/MyDrive/10k_processed/chunk_meta.parquet")
assert p.exists(), f"File not found: {p}"

df = pd.read_parquet(p)

print(df.shape)
df.head()

(8817, 6)


,id,doc_id,filename,page_start,page_end,text_clean
0,0,9cf140658edcf90c20d3825c90fd9c0f,3M_10K.pdf,1,1,Table of Contents UNITED STATES SECURITIES AND...
1,1,9cf140658edcf90c20d3825c90fd9c0f,3M_10K.pdf,1,1,Yes o No x Indicate by check mark whether the ...
2,2,9cf140658edcf90c20d3825c90fd9c0f,3M_10K.pdf,1,3,o Indicate by check mark whether any of those ...
3,3,9cf140658edcf90c20d3825c90fd9c0f,3M_10K.pdf,3,3,Cybersecurity 17 Item 2. Properties 18 Item 3....
4,4,9cf140658edcf90c20d3825c90fd9c0f,3M_10K.pdf,3,3,Financial Statements and Supplementary Data 41...


In [3]:
import os, math, io
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from transformers import AutoTokenizer, AutoModel

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if (device=="cuda" and torch.cuda.get_device_capability(0)[0] >= 8) else torch.float32
device, dtype


('cuda', torch.bfloat16)

In [4]:
MODEL_ID = "Qwen/Qwen3-Embedding-8B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModel.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    torch_dtype=dtype,
    device_map="auto" if device=="cuda" else None
)
model.eval()


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/729 [00:00<?, ?B/s]

...

model-00001-of-00004.safetensors:   0%|          | 0.00/4.90G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/336M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Qwen3Model(
  (embed_tokens): Embedding(151665, 4096)
  (layers): ModuleList(
    (0-35): 36 x Qwen3DecoderLayer(
      (self_attn): Qwen3Attention(
        (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
        (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
        (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
        (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
        (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
      )
      (mlp): Qwen3MLP(
        (gate_proj): Linear(in_features=4096, out_features=12288, bias=False)
        (up_proj): Linear(in_features=4096, out_features=12288, bias=False)
        (down_proj): Linear(in_features=12288, out_features=4096, bias=False)
        (act_fn): SiLU()
      )
      (input_layernorm): Qwen3RMSNorm((4096,), eps=1e-06)
      (post_attention_layernorm): Qwen3RMSNorm((4096,), eps=1e-06)
    )
  )
  (norm): Qwen3RMSNorm((

In [5]:
import torch
import torch.nn.functional as F

def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts


In [6]:
def embed_texts(texts, batch_size=16, max_length=768, device=None):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    all_embs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )
        enc = {k: v.to(device) for k, v in enc.items()}

        with torch.no_grad():
            out = model(**enc)
            pooled = mean_pool(out.last_hidden_state, enc["attention_mask"])
            pooled = F.normalize(pooled, p=2, dim=1)

        all_embs.append(pooled.cpu())

    return torch.cat(all_embs, dim=0).numpy()


In [7]:

assert "text_clean" in df.columns, "text_clean column not found in df"

texts = df["text_clean"].fillna("").astype(str).tolist()
len(texts), df.shape


(8817, (8817, 6))

In [9]:
from tqdm.auto import tqdm
import numpy as np
import torch
import torch.nn.functional as F

def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts

def embed_texts_progress(
    texts,
    batch_size=16,
    max_length=512,
    device=None,
    save_every=None,
    save_path=None
):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    n = len(texts)
    all_chunks = []
    n_batches = (n + batch_size - 1) // batch_size

    for bi in tqdm(range(n_batches), desc="Embedding", unit="batch"):
        s = bi * batch_size
        e = min(s + batch_size, n)
        batch = texts[s:e]

        enc = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )
        enc = {k: v.to(device) for k, v in enc.items()}

        with torch.no_grad():
            out = model(**enc)
            pooled = mean_pool(out.last_hidden_state, enc["attention_mask"])
            pooled = F.normalize(pooled, p=2, dim=1)

        all_chunks.append(pooled.cpu().numpy())


        if save_every and save_path and ((bi + 1) % save_every == 0):
            np.save(save_path, np.vstack(all_chunks))

    return np.vstack(all_chunks)


In [12]:
BATCH_SIZE = 32
MAX_LEN = 1024

embeddings = embed_texts_progress(
    texts,
    batch_size=BATCH_SIZE,
    max_length=MAX_LEN,
    device="cuda" if torch.cuda.is_available() else "cpu",
    save_every=100,
    save_path="/content/drive/MyDrive/10k_processed/qwen3_emb.npy"
)
embeddings.shape


Embedding:   0%|          | 0/276 [00:00<?, ?batch/s]

(8817, 4096)

In [16]:
import numpy as np
import faiss

emb_mat = np.asarray(embeddings, dtype="float32")


index = faiss.IndexFlatIP(emb_mat.shape[1])
index.add(emb_mat)
print("Indexed vectors:", index.ntotal, "| dim:", emb_mat.shape[1])


Indexed vectors: 8817 | dim: 4096


In [17]:
from pathlib import Path
import pandas as pd

save_dir = Path("/content/drive/MyDrive/10k_processed")
save_dir.mkdir(parents=True, exist_ok=True)

index_path = str(save_dir / "qwen3_index_flatip.faiss")
faiss.write_index(index, index_path)

##Save Faiss
idmap_path = str(save_dir / "qwen3_index_ids.csv")
pd.Series(df.index, name="row_id").to_csv(idmap_path, index=False)

print("Saved index to:", index_path)
print("Saved id map to:", idmap_path)


Saved index to: /content/drive/MyDrive/10k_processed/qwen3_index_flatip.faiss
Saved id map to: /content/drive/MyDrive/10k_processed/qwen3_index_ids.csv


In [19]:
import pandas as pd

qa_data = [
    {
        "question": "What major risks does Alphabet cite from operating internationally and from financial exposures?",
        "answer": "Alphabet flags anti-bribery compliance ... some acquisition assets/liabilities—require subjective fair-value estimates."
    },
    {
        "question": "Summarize Alan R. Mulally’s service on the company’s board and his prior leadership at Ford and Boeing, including dates, committee memberships, and key advisory roles.",
        "answer": "Alan R. Mulally has served on the company’s Board of Directors since July 2014 ... U.S. Air Force Scientific Advisory Board."
    },
    {
        "question": "Summarize the key governance and career details mentioned: John’s planned resignation from Stanford, Ann Mather’s current board seats and committee roles, her prior directorships and executive experience, her education and credential, and Alan R. Mulally’s board service with dates.",
        "answer": "John announced he will resign as President of Stanford University in August 2016 ... Alan R. Mulally has served on our Board since July 2014."
    },
    {
        "question": "In brief, what does Ford’s global facilities footprint and ownership/lease mix look like as of Dec 31, 2024?",
        "answer": "Mostly leased warehouses and sales offices ... 41 manufacturing/assembly plants supporting Ford Blue, Model e, and Ford Pro."
    },
    {
        "question": "Briefly explain how Microsoft measures fair value for Level 2 and Level 3 items, how it values equity investments without readily determinable fair values, and its policy for property and equipment.",
        "answer": "Level 2 uses observable inputs ... Property and equipment is at cost, depreciated straight-line ..."
    },
    {
        "question": "What macroeconomic and regulatory factors could materially impact Walmart, and how might they affect Walmart’s demand, margins, costs, inventory, suppliers, and partnerships?",
        "answer": "Higher rates and energy costs, inflation/deflation ... and cause partnerships/alliances to underperform."
    },
    {
        "question": "In brief, what were Pfizer’s key 2024 items (gains, dividends, charges) and what intangible-asset impairments did it record, including fair-value levels and valuation method?",
        "answer": "In 2024 Pfizer booked $945M gains ... measured via the income approach."
    },
    {
        "question": "What does Ford’s 2024 Form 10-K cover and how is the company classified, including listing details?",
        "answer": "Ford filed a Form 10-K for the year ended Dec 31, 2024 ... Listed on the NYSE ..."
    },
]

qa_df = pd.DataFrame(qa_data)
qa_df.head(2)


,question,answer
0,What major risks does Alphabet cite from opera...,Alphabet flags anti-bribery compliance ... som...
1,Summarize Alan R. Mulally’s service on the com...,Alan R. Mulally has served on the company’s Bo...


In [24]:
import torch
import torch.nn.functional as F
import re

def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    return (last_hidden_state * mask).sum(1) / torch.clamp(mask.sum(1), min=1e-9)

def retrieve_texts(question: str, top_k=5, max_len=1024):
    enc = tokenizer(question, return_tensors="pt", truncation=True, max_length=max_len)
    enc = {k: v.to(next(model.parameters()).device) for k, v in enc.items()}
    with torch.no_grad():
        out = model(**enc)
        q = mean_pool(out.last_hidden_state, enc["attention_mask"])
        q = F.normalize(q, p=2, dim=1).cpu().numpy().astype("float32")
    D, I = index.search(q, top_k)

    ctx = df.iloc[I[0]][["text_clean"]].fillna("").astype(str).tolist()
    scores = D[0].tolist()
    return ctx, scores

def extractive_answer(question: str, contexts: list[str], max_sentences=5) -> str:

    text = (contexts[0] if contexts else "").strip()
    sents = re.split(r'(?<=[.!?])\s+', text)
    return " ".join(sents[:max_sentences]).strip()


In [26]:
def retrieve_texts(question: str, top_k=5, max_len=1024):
    import torch, torch.nn.functional as F
    enc = tokenizer(question, return_tensors="pt", truncation=True, max_length=max_len)
    enc = {k: v.to(next(model.parameters()).device) for k, v in enc.items()}

    with torch.no_grad():
        out = model(**enc)
        q = mean_pool(out.last_hidden_state, enc["attention_mask"])
        q = F.normalize(q, p=2, dim=1).cpu().numpy().astype("float32")

    D, I = index.search(q, top_k)

    ctx_series = df["text_clean"].iloc[I[0]].fillna("").astype(str)
    ctx = ctx_series.tolist()
    scores = D[0].tolist()
    return ctx, scores


In [27]:
preds = []
for q in qa_df["question"]:
    ctx, _ = retrieve_texts(q, top_k=5, max_len=1024)
    pred = extractive_answer(q, ctx, max_sentences=5)
    preds.append(pred)

qa_df = qa_df.copy()
qa_df["predicted_answer"] = preds
qa_df.head(2)


,question,answer,predicted_answer
0,What major risks does Alphabet cite from opera...,Alphabet flags anti-bribery compliance ... som...,"In addition, our products and services are hig..."
1,Summarize Alan R. Mulally’s service on the com...,Alan R. Mulally has served on the company’s Bo...,Alan R. Mulally has served as a member of our ...


In [28]:
import json, re, time, requests
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

OPENAI_MODEL = "gpt-4o-mini"

SYSTEM_PROMPT = """You are an exacting evaluator for finance Q&A.
Score the model's answer against the reference answer using this rubric:
- 5 = Fully correct and complete, matches key facts and reasoning.
- 4 = Mostly correct, minor omissions/wording differences.
- 3 = Partially correct, noticeable gaps or minor inaccuracies.
- 2 = Mostly incorrect, only small parts overlap.
- 1 = Barely related, largely wrong.
- 0 = Completely wrong or irrelevant.

Return strict JSON: {"score": int(0-5), "rationale": "short explanation"}.
Do not include extra keys or commentary.
"""

def _force_json(s: str) -> dict:

    try:
        return json.loads(s)
    except Exception:
        pass

    m = re.search(r"\{.*\}", s, flags=re.S)
    if m:
        try:
            return json.loads(m.group(0))
        except Exception:
            pass
    raise ValueError("Judge did not return valid JSON.\nGot:\n" + s)

@retry(
    reraise=True,
    stop=stop_after_attempt(5),
    wait=wait_exponential(multiplier=1, min=1, max=20),
    retry=retry_if_exception_type((requests.RequestException, ValueError))
)
def judge_one(question: str, reference: str, prediction: str) -> dict:
    headers = {
        "Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}",
        "Content-Type": "application/json",
    }
    user_prompt = f"""
Question:
{question}

Reference answer:
{reference}

Model answer:
{prediction}

Return only JSON with "score" and "rationale".
""".strip()

    payload = {
        "model": OPENAI_MODEL,
        "temperature": 0,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ]
    }

    resp = requests.post(OPENAI_URL, headers=headers, json=payload, timeout=60)
    resp.raise_for_status()
    text = resp.json()["choices"][0]["message"]["content"]
    return _force_json(text)


In [32]:
import re
import torch
import torch.nn.functional as F

def retrieve_texts(question: str, top_k=5, max_len=1024):
    enc = tokenizer(question, return_tensors="pt", truncation=True, max_length=max_len)
    enc = {k: v.to(next(model.parameters()).device) for k, v in enc.items()}
    with torch.no_grad():
        out = model(**enc)
        q = mean_pool(out.last_hidden_state, enc["attention_mask"])
        q = F.normalize(q, p=2, dim=1).cpu().numpy().astype("float32")
    D, I = index.search(q, top_k)


    ctx_series = df["text_clean"].iloc[I[0]].fillna("").astype(str)
    ctx = ctx_series.tolist()
    return ctx

qa_df = qa_df.copy()
qa_df["contexts"] = [retrieve_texts(q, top_k=5, max_len=1024) for q in qa_df["question"]]


qa_df["ground_truth"] = qa_df["answer"].fillna("").astype(str)
qa_df["answer"] = qa_df["predicted_answer"].fillna("").astype(str)


In [33]:
from datasets import Dataset


def ensure_list(x):
    if isinstance(x, list): return x
    if x is None: return []
    return [str(x)]

ragas_ds = Dataset.from_dict({
    "question": qa_df["question"].astype(str).tolist(),
    "contexts": [ensure_list(c) for c in qa_df["contexts"].tolist()],
    "answer": qa_df["answer"].astype(str).tolist(),
    "ground_truth": qa_df["ground_truth"].astype(str).tolist()
})
ragas_ds


Dataset({
    features: ['question', 'contexts', 'answer', 'ground_truth'],
    num_rows: 8
})

In [38]:
import torch
import torch.nn.functional as F
from typing import List
from langchain_core.embeddings import Embeddings


def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    return (last_hidden_state * mask).sum(1) / torch.clamp(mask.sum(1), min=1e-9)

class QwenEmbeddings(Embeddings):
    def __init__(self, tokenizer, model, max_length: int = 512, batch_size: int = 16, device: str = None):
        self.tokenizer = tokenizer
        self.model = model
        self.max_length = max_length
        self.batch_size = batch_size
        if device is None:
            device = "cuda" if torch.cuda.is_available() else "cpu"
        self.device = device

    def _embed_texts(self, texts: List[str]) -> List[List[float]]:
        vecs = []
        for i in range(0, len(texts), self.batch_size):
            batch = [t if isinstance(t, str) else "" for t in texts[i:i+self.batch_size]]
            enc = self.tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=self.max_length,
                return_tensors="pt"
            )
            enc = {k: v.to(self.device) for k, v in enc.items()}
            with torch.no_grad():
                out = self.model(**enc)
                pooled = mean_pool(out.last_hidden_state, enc["attention_mask"])
                pooled = F.normalize(pooled, p=2, dim=1)
            vecs.append(pooled.cpu().tolist())

        return [v for chunk in vecs for v in chunk]


    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        return self._embed_texts(texts)

    def embed_query(self, text: str) -> List[float]:
        return self._embed_texts([text])[0]


In [42]:

qwen_emb = QwenEmbeddingsLC(
    tokenizer=tokenizer,
    hf_model=model,
    model_id=MODEL_ID,
    max_length=512,
    batch_size=16,
)

from ragas import evaluate
from ragas.metrics import (
    answer_relevancy,
    faithfulness,
    context_precision,
    context_recall,
    answer_correctness,
)
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

result = evaluate(
    ragas_ds,
    metrics=[answer_relevancy, faithfulness, context_precision, context_recall, answer_correctness],
    llm=llm,
    embeddings=qwen_emb,
)

ragas_df = result.to_pandas()
ragas_df.head()


Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

,user_input,retrieved_contexts,response,reference,answer_relevancy,faithfulness,context_precision,context_recall,answer_correctness
0,What major risks does Alphabet cite from opera...,"[In addition, our products and services are hi...","In addition, our products and services are hig...",Alphabet flags anti-bribery compliance ... som...,0.000000,1.0,0.000,0.5,0.178091
1,Summarize Alan R. Mulally’s service on the com...,[Alan R. Mulally has served as a member of our...,Alan R. Mulally has served as a member of our ...,Alan R. Mulally has served on the company’s Bo...,0.832911,1.0,1.000,1.0,0.395894
2,Summarize the key governance and career detail...,[John has announced that he plans to resign fr...,John has announced that he plans to resign fro...,John announced he will resign as President of ...,0.779735,1.0,1.000,1.0,0.321281
3,"In brief, what does Ford’s global facilities f...",[The majority of the warehouses that we operat...,The majority of the warehouses that we operate...,Mostly leased warehouses and sales offices ......,0.792985,1.0,0.700,1.0,0.617007
4,Briefly explain how Microsoft measures fair va...,[Investments that are measured at fair value u...,Investments that are measured at fair value us...,Level 2 uses observable inputs ... Property an...,0.810792,1.0,0.325,0.5,0.419906


In [44]:
wanted = {"answer_relevancy","faithfulness","context_precision","context_recall","answer_correctness"}
metric_cols = [c for c in ragas_df.columns if c in wanted]

metrics_df = ragas_df[metric_cols].copy()
metrics_df.head(8)


,answer_relevancy,faithfulness,context_precision,context_recall,answer_correctness
0,0.000000,1.0,0.000000,0.5,0.178091
1,0.832911,1.0,1.000000,1.0,0.395894
2,0.779735,1.0,1.000000,1.0,0.321281
3,0.792985,1.0,0.700000,1.0,0.617007
4,0.810792,1.0,0.325000,0.5,0.419906
5,0.000000,1.0,0.477778,1.0,0.208724
6,0.000000,1.0,0.333333,0.0,0.267600
7,0.820736,1.0,1.000000,1.0,0.186393
